In [ ]:
from flask import Flask, jsonify, request, render_template
from flask_cors import CORS
from werkzeug.serving import run_simple
from datetime import datetime
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

app = Flask(__name__)
CORS(app)

# Data
vendors = [
    {"vendor_id": "V001", "vendor_name": "Fresh Orchard Co.", "contact": "+95-1-234567", "location": "Yangon"},
    {"vendor_id": "V002", "vendor_name": "Tropical Fruits Hub", "contact": "+95-9-888777", "location": "Mandalay"},
    {"vendor_id": "V003", "vendor_name": "Golden Harvest Fruits", "contact": "+95-9-123456", "location": "Naypyitaw"},
    {"vendor_id": "V004", "vendor_name": "Juicy Basket Mart", "contact": "+95-9-654321", "location": "Yangon"}
]


items = [
    {"item_id": "I001", "item_name": "Apple", "qty": 120, "price": 1500, "vendor_id": "V001"},
    {"item_id": "I002", "item_name": "Banana", "qty": 200, "price": 500, "vendor_id": "V001"},
    {"item_id": "I003", "item_name": "Orange", "qty": 180, "price": 1000, "vendor_id": "V002"},
    {"item_id": "I004", "item_name": "Mango", "qty": 100, "price": 2000, "vendor_id": "V002"},
    {"item_id": "I005", "item_name": "Grapes", "qty": 150, "price": 1800, "vendor_id": "V003"},
    {"item_id": "I006", "item_name": "Pineapple", "qty": 70, "price": 2500, "vendor_id": "V003"},
    {"item_id": "I007", "item_name": "Watermelon", "qty": 60, "price": 3000, "vendor_id": "V004"},
    {"item_id": "I008", "item_name": "Strawberry", "qty": 90, "price": 2200, "vendor_id": "V004"},
    {"item_id": "I009", "item_name": "Blueberry", "qty": 50, "price": 3500, "vendor_id": "V001"},
    {"item_id": "I010", "item_name": "Kiwi", "qty": 80, "price": 1800, "vendor_id": "V001"},
    {"item_id": "I011", "item_name": "Papaya", "qty": 60, "price": 1700, "vendor_id": "V002"},
    {"item_id": "I012", "item_name": "Guava", "qty": 100, "price": 1000, "vendor_id": "V002"},
    {"item_id": "I013", "item_name": "Peach", "qty": 70, "price": 2000, "vendor_id": "V003"},
    {"item_id": "I014", "item_name": "Pear", "qty": 90, "price": 1600, "vendor_id": "V003"},
    {"item_id": "I015", "item_name": "Plum", "qty": 80, "price": 1500, "vendor_id": "V004"},
    {"item_id": "I016", "item_name": "Cherry", "qty": 60, "price": 2800, "vendor_id": "V004"},
    {"item_id": "I017", "item_name": "Lychee", "qty": 100, "price": 2000, "vendor_id": "V001"},
    {"item_id": "I018", "item_name": "Dragon Fruit", "qty": 50, "price": 3000, "vendor_id": "V002"},
    {"item_id": "I019", "item_name": "Pomegranate", "qty": 110, "price": 2500, "vendor_id": "V003"},
    {"item_id": "I020", "item_name": "Coconut", "qty": 130, "price": 1800, "vendor_id": "V004"}
]


orders = [
    {"order_id": "O001", "item_id": "I001", "qty": 10, "vendor_id": "V001", "order_date": "2025-08-01", "received_date": "2025-08-03"},
    {"order_id": "O002", "item_id": "I002", "qty": 15, "vendor_id": "V001", "order_date": "2025-08-10", "received_date": "2025-08-13"},
    {"order_id": "O003", "item_id": "I003", "qty": 25, "vendor_id": "V004", "order_date": "2025-08-15", "received_date": None},
    {"order_id": "O004", "item_id": "I005", "qty": 40, "vendor_id": "V004", "order_date": "2025-08-02", "received_date": "2025-08-05"}
]

payments = [
    {"payment_id": "P001", "customer_id": "C001", "item_id": ["I001", "I002"], "qty": [30, 40], "amount": 65000,"address": "pyay","phno": "09683969002","method": "payment", "status": "COMFIRM", "date": "2025-08-01"},
    {"payment_id": "P002", "customer_id": "C001", "item_id": ["I003", "I004"], "qty": [16, 17], "amount": 50000,"address": "pyay","phno": "09683969002", "method": "Loan", "status": "COMFIRM", "date": "2025-08-02"},
    {"payment_id": "P003", "customer_id": "C002", "item_id": ["I011", "I010"], "qty": [10, 20], "amount": 53000,"address": "pyay","phno": "09683969002", "method": "payment","status": "Order","date": "2025-08-02"},
    {"payment_id": "P004", "customer_id": "C002","item_id": ["I014", "I020"], "qty": [30, 40], "amount": 120000,"address": "pyay","phno": "09683969002", "method": "Loan", "status": "Order", "date": "2025-08-02"}
]

loans = [
    {"loan_id": "L001", "payment_id": "P002", "name": "Kevin", "age": 30, "annual_income": 300000, "marital_status": "single", "education": "Bachelor", "credit_score": 750},
    {"loan_id": "L002", "payment_id": "P004", "name": "Marshal", "age": 29, "annual_income": 250000, "marital_status": "married", "education": "Master", "credit_score": 500}
]


# View for html code------------------------------------------------------------------------------------------------------------

@app.route('/')
def index():
    return render_template("Dashboard.html")

@app.route('/viewDashboard')
def viewDashboard():
    return render_template("Dashboard.html")

@app.route('/viewVendors')
def viewVendors():
    return render_template("Vendors.html")

@app.route('/viewInventory')
def viewInventory():
    return render_template("Inventory.html")

@app.route('/viewOrders')
def viewOrders():
    return render_template("Orders.html")

@app.route('/viewCustomer')
def viewCustomer():
    return render_template("customerOrders.html")

# API---------------------------------------------------------------------------------------------------------------------------
@app.route('/api/vendors', methods=['GET'])
def get_vendors():
    return jsonify(vendors)

@app.route('/api/items', methods=['GET'])
def get_items():
    return jsonify(items)

@app.route('/api/orders', methods=['GET'])
def get_orders():
    return jsonify(orders)

@app.route('/api/payments', methods=['GET'])
def get_payments():
    return jsonify(payments)

@app.route('/api/loans', methods=['GET'])
def get_loans():
    return jsonify(loans)

@app.route('/api/dashboard', methods=['GET'])
def get_dashboard_data():
    users = len(set(payment['customer_id'] for payment in payments))

    revenue = sum(p['amount'] for p in payments if p['status'] == "COMFIRM")

    new_orders = sum(1 for order in orders if order['received_date'] is None)

    messages = 3  

    inventory_alerts = sum(1 for item in items if item['qty'] < 80)

    system_status = "All systems operational. No issues reported in the last 24 hours."

    return jsonify({
        "users": users,
        "revenue": revenue,
        "newOrders": new_orders,
        "messages": messages,
        "inventoryAlerts": inventory_alerts,
        "systemStatus": system_status
    })


# Function for Vendors----------------------------------------------------------------------------------------------------------

@app.route('/addVendors', methods=['POST'])
def addVendors():
    try:
        data = request.get_json()
        vendor_name = data.get('vendor_name')
        contact = data.get('contact')
        location = data.get('location')

        if not vendor_name or not contact or not location:
            return jsonify({"error": "Missing required fields"}), 400

        existing_ids = [int(v['vendor_id'][1:]) for v in vendors if v['vendor_id'][1:].isdigit()]
        new_id_num = max(existing_ids, default=0) + 1
        new_vendor_id = f"V{new_id_num:03d}"

        new_vendor = {
            "vendor_id": new_vendor_id,
            "vendor_name": vendor_name,
            "contact": contact,
            "location": location
        }

        vendors.append(new_vendor)
        return render_template("Vendors.html")

    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/editVendors', methods=['POST'])
def editVendors():
    try:
        data = request.get_json()
        vendor_id = data.get('vendor_id')
        vendor_name = data.get('vendor_name')
        contact = data.get('contact')
        location = data.get('location')

        if not vendor_id or not vendor_name or not contact or not location:
            return jsonify({"error": "Missing required fields"}), 400

        for vendor in vendors:
            if vendor['vendor_id'] == vendor_id:
                vendor['vendor_name'] = vendor_name
                vendor['contact'] = contact
                vendor['location'] = location
                return render_template("Vendors.html")

        return jsonify({"error": "Vendor not found"}), 404

    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/removeVendors', methods=['POST'])
def removeVendors():
    try:
        vendor_id = request.form.get('vendor_id')

        if not vendor_id:
            return jsonify({"error": "Missing vendor_id"}), 400

        for i, vendor in enumerate(vendors):
            if vendor['vendor_id'] == vendor_id:
                vendors.pop(i)
                return render_template("Vendors.html")

        return jsonify({"error": "Vendor not found"}), 404

    except Exception as e:
        return jsonify({"error": str(e)}), 500

# Function for Inventory--------------------------------------------------------------------------------------------------------

@app.route('/addInventory', methods=['POST'])
def addInventory():
    try:
        data = request.get_json()

        item_name = data.get('item_name')
        qty = int(data.get('qty'))
        price = float(data.get('price'))
        vendor_id = data.get('vendor_id')

        if not item_name or qty <= 0 or price <= 0 or not vendor_id:
            return jsonify({"error": "Invalid input data"}), 400

        existing_item_ids = [int(i['item_id'][1:]) for i in items if i['item_id'][1:].isdigit()]
        new_item_id = f"I{(max(existing_item_ids, default=0) + 1):03d}"

        new_item = {
            "item_id": new_item_id,
            "item_name": item_name,
            "qty": 0,
            "price": price,
            "vendor_id": vendor_id
        }
        items.append(new_item)

        existing_order_ids = [int(o['order_id'][1:]) for o in orders if o['order_id'][1:].isdigit()]
        new_order_id = f"O{(max(existing_order_ids, default=0) + 1):03d}"

        new_order = {
            "order_id": new_order_id,
            "item_id": new_item_id,
            "qty": qty,  # From user
            "vendor_id": vendor_id,
            "order_date": datetime.today().strftime("%Y-%m-%d"),
            "received_date": None
        }
        orders.append(new_order)

        return jsonify({
            "message": "Item and order added successfully.",
            "item": new_item,
            "order": new_order
        }), 201

    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/orderInventory', methods=['POST'])
def orderInventory():
    try:
        data = request.get_json()

        item_id = data.get('item_id')
        qty = int(data.get('qty'))
        vendor_id = data.get('vendor_id')

        if not item_id or qty <= 0 or not vendor_id:
            return jsonify({"error": "Invalid input data"}), 400

        existing_order_ids = [int(o['order_id'][1:]) for o in orders if o['order_id'][1:].isdigit()]
        new_order_id = f"O{(max(existing_order_ids, default=0) + 1):03d}"

        new_order = {
            "order_id": new_order_id,
            "item_id": item_id,
            "qty": qty,
            "vendor_id": vendor_id,
            "order_date": datetime.today().strftime("%Y-%m-%d"),
            "received_date": None
        }

        orders.append(new_order)

        return jsonify({
            "message": "Order placed successfully.",
            "order": new_order
        }), 201

    except Exception as e:
        return jsonify({"error": str(e)}), 500

    
@app.route('/editInventory', methods=['POST'])
def editInventory():
    try:
        data = request.get_json()

        item_id = data.get('item_id')
        item_name = data.get('item_name')
        qty = int(data.get('qty'))
        price = float(data.get('price'))

        if not item_id or not item_name or qty < 0 or price < 0:
            return jsonify({"error": "Invalid input data"}), 400

        for item in items:
            if item['item_id'] == item_id:
                item['item_name'] = item_name
                item['qty'] = qty
                item['price'] = price
                return jsonify({
                    "message": "Item updated successfully.",
                    "item": item
                }), 200

        return jsonify({"error": "Item not found"}), 404

    except Exception as e:
        return jsonify({"error": str(e)}), 500

# function for Order------------------------------------------------------------------------------------------------------------

@app.route('/receivedOrder', methods=['POST'])
def receivedOrder():
    order_id = request.form.get('order_id')
    item_id = request.form.get('item_id')

    for order in orders:
        if order['order_id'] == order_id and order['item_id'] == item_id:
            order['received_date'] = datetime.today().strftime("%Y-%m-%d")
            
            for item in items:
                if item['item_id'] == item_id:
                    item['qty'] += order['qty']
            break

    return render_template("Orders.html")

# customer order----------------------------------------------------------------------------------------------------------------

@app.route('/newPayment', methods=['POST'])
def newpayment():
    data = request.get_json()
    payments.append(data)
    return jsonify({})
    
@app.route('/newLoans', methods=['POST'])
def newLoans():
    data = request.get_json()
    loans.append(data)
    return jsonify({})

@app.route('/api/payment/<payment_id>', methods=['GET'])
def get_payment(payment_id):
    # find the payment
    payment = next((p for p in payments if p['payment_id'] == payment_id), None)
    if not payment:
        return jsonify({"error": "Payment not found"}), 404
    return jsonify(payment)

@app.route('/api/loan_by_payment/<payment_id>', methods=['GET'])
def get_loan_by_payment(payment_id):
    # find a loan record with that payment_id
    loan = next((l for l in loans if l['payment_id'] == payment_id), None)
    if not loan:
        return jsonify({"error": "Loan not found"}), 404
    return jsonify(loan)

@app.route('/api/confirmPayment', methods=['POST'])
def confirm_payment():
    data = request.get_json()
    payment_id = data.get('payment_id')
    
    payment = next((p for p in payments if p['payment_id'] == payment_id), None)
    
    if not payment:
        return jsonify({"error": "Invalid payment_id"}), 400
    
    payment['status'] = "CONFIRM"
    
    for item_id, qty_purchased in zip(payment.get('item_id', []), payment.get('qty', [])):
        item = next((i for i in items if i['item_id'] == item_id), None)
        if item:
            item['qty'] = max(item.get('qty', 0) - qty_purchased, 0)
    
    return jsonify({}), 200



@app.route('/api/rejectPayment', methods=['POST'])
def reject_payment():
    data = request.get_json()
    payment_id = data.get('payment_id')
    
    payment = next((p for p in payments if p['payment_id'] == payment_id), None)
    
    if not payment:
        return jsonify({"error": "Invalid payment_id"}), 400
    
    payment['status'] = "REJECT"
    return jsonify({}), 200

# ML-----------------------------------------------------------------------------------------------------------------------------

X = np.array([
    [25, 50000, 700, 1, 3],
    [40, 120000, 800, 0, 4],
    [35, 70000, 650, 1, 2],
    [50, 100000, 780, 0, 4],
    [28, 45000, 600, 1, 1]
])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=2, random_state=42)
kmeans.fit(X_scaled)

def loan_unsupervised_predict(user_form):
    base_score = 0
    if user_form['credit_score'] > 650:
        base_score += 0.4
    if user_form['annual_income'] > 60000:
        base_score += 0.3
    if user_form['age'] > 25:
        base_score += 0.2

        
    marital_status_mapping = {
        "single": 0,
        "married": 1,
    }

    education_level_mapping = {
        "High School": 0,
        "Diploma": 1,
        "Bachelor": 2,
        "Master": 3,
    }
    user_form['married_status'] = marital_status_mapping[user_form['married_status']]
    user_form['education_level'] = education_level_mapping[user_form['education_level']]
    features = np.array([
        [
            user_form['age'],
            user_form['annual_income'],
            user_form['credit_score'],
            user_form['married_status'],
            user_form['education_level']
        ]
    ])
    features_scaled = scaler.transform(features)

    cluster = kmeans.predict(features_scaled)[0]
    if cluster == 1:
        base_score += 0.2

    approval = base_score > 0.7

    return {
        "loan_approval": approval,
        "repayment_probability": round(min(base_score, 1.0) * 100, 2),
        "cluster": 1
    }

@app.route('/api/testLoans', methods=['POST'])
def testLoans():
    data = request.json

    if not data or 'loan_id' not in data:
        return jsonify({"error": "Missing loan_id in request"}), 400

    loan_id = data['loan_id']

    loan_data = next((loan for loan in loans if loan['loan_id'] == loan_id), None)
    if not loan_data:
        return jsonify({"error": f"Loan with id '{loan_id}' not found"}), 404

    payload = {
        "age": loan_data['age'],
        "annual_income": loan_data['annual_income'],
        "married_status": loan_data['marital_status'],
        "education_level": loan_data['education'],
        "credit_score": loan_data['credit_score']
    }

    required = ['age', 'annual_income', 'married_status', 'education_level', 'credit_score']
    if not all(k in payload for k in required):
        return jsonify({"error": "Loan data missing required fields"}), 400

    try:
        result = loan_unsupervised_predict(payload)
        return jsonify(result)
    except Exception as e:
        return jsonify({"error": str(e)}), 500

# Run the app-------------------------------------------------------------------------------------------------------------------

if __name__ == '__main__':
    run_simple("192.168.181.241", 4001, app, use_reloader=False)


 * Running on http://192.168.181.241:4001
Press CTRL+C to quit
192.168.181.241 - - [12/Sep/2025 15:04:03] "GET /api/items HTTP/1.1" 200 -
192.168.181.241 - - [12/Sep/2025 15:04:27] "GET / HTTP/1.1" 200 -
192.168.181.241 - - [12/Sep/2025 15:04:27] "GET /api/dashboard HTTP/1.1" 200 -
192.168.181.241 - - [12/Sep/2025 15:04:27] "GET /api/items HTTP/1.1" 200 -
192.168.181.241 - - [12/Sep/2025 15:04:27] "GET /api/orders HTTP/1.1" 200 -
192.168.181.241 - - [12/Sep/2025 15:04:27] "GET /api/payments HTTP/1.1" 200 -
192.168.181.241 - - [12/Sep/2025 15:04:27] "GET /favicon.ico HTTP/1.1" 404 -
192.168.181.21 - - [12/Sep/2025 15:05:42] "GET /api/items HTTP/1.1" 200 -
192.168.181.21 - - [12/Sep/2025 15:05:54] "GET /api/items HTTP/1.1" 200 -
192.168.181.241 - - [12/Sep/2025 15:09:26] "GET /viewVendors HTTP/1.1" 200 -
192.168.181.241 - - [12/Sep/2025 15:09:26] "GET /api/vendors HTTP/1.1" 200 -
192.168.181.241 - - [12/Sep/2025 15:09:27] "GET /viewDashboard HTTP/1.1" 200 -
192.168.181.241 - - [12/Sep/20